# 08 — Siamese Pretrained Inference

**Learning goal:** Run a pretrained Siamese change detection network on Marshall Fire
SAR patches *before* any fine-tuning. This establishes the zero-shot baseline.

**Key question:** What does the pretrained model predict before any fine-tuning?
We expect mediocre performance (~0.55-0.70 AUC) because the model has never seen
fire damage in SAR data. Fine-tuning (notebook 09) should improve this.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rasterio
import torch
import torch.nn as nn

AOI_BBOX = [-105.23, 39.915, -105.12, 39.98]
DATA_DIR = Path("../data")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Catalog connected.")

## Define Patch Extraction Helper

Extract a small (64x64 pixel) patch centered on a given coordinate from
Sentinel-1 GRD VV and VH bands. Returns a tensor of shape `(2, H, W)`.

In [ ]:
def extract_patch(item, centroid_lon, centroid_lat, patch_size=64):
    """
    Extract a (2, patch_size, patch_size) tensor from Sentinel-1 VV+VH bands.

    Parameters
    ----------
    item : pystac.Item
        Sentinel-1 GRD item from Planetary Computer.
    centroid_lon, centroid_lat : float
        Center coordinates in EPSG:4326.
    patch_size : int
        Output patch size in pixels.

    Returns
    -------
    torch.Tensor of shape (2, patch_size, patch_size)
        Channel 0 = VV (dB), Channel 1 = VH (dB)
    """
    bands = []
    for band_name in ["vv", "vh"]:
        href = planetary_computer.sign(item.assets[band_name].href)
        with rasterio.open(href) as src:
            # Convert centroid to the raster's native CRS
            if src.crs != "EPSG:4326":
                from pyproj import Transformer
                transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                cx, cy = transformer.transform(centroid_lon, centroid_lat)
            else:
                cx, cy = centroid_lon, centroid_lat

            row, col = src.index(cx, cy)
            half = patch_size // 2

            # Ensure window is within raster bounds
            row_start = max(0, row - half)
            col_start = max(0, col - half)
            window = rasterio.windows.Window(
                col_off=col_start, row_off=row_start,
                width=patch_size, height=patch_size
            )

            data = src.read(1, window=window).astype(np.float32)

            # Convert linear power to dB
            data = np.where(data > 0, 10 * np.log10(data), -30.0)

            # Pad if patch is at the edge of the raster
            if data.shape != (patch_size, patch_size):
                padded = np.full((patch_size, patch_size), -30.0, dtype=np.float32)
                padded[:data.shape[0], :data.shape[1]] = data
                data = padded

            bands.append(data)

    # Stack VV + VH -> (2, H, W)
    tensor = torch.from_numpy(np.stack(bands, axis=0))
    return tensor


print("extract_patch() defined.")
print("Output shape: (2, patch_size, patch_size) — channels are [VV_dB, VH_dB]")

## Define Simple Siamese Change Detection Model

Architecture:
1. **Encoder** — Modified ResNet50 backbone (first conv changed to accept 2 channels)
2. **Siamese comparison** — absolute difference of pre/post feature maps
3. **Classification head** — linear layers reducing to a single change probability
4. **Sigmoid** — output in [0, 1]

The model processes pre-fire and post-fire patches through the same encoder,
then computes the absolute difference of feature vectors to detect change.

In [ ]:
import torchvision.models as models


class SiameseChangeDetector(nn.Module):
    """Siamese change detection network using ResNet50 backbone.

    Takes pre and post patches (each 2-channel: VV+VH) and outputs
    a change probability score.
    """

    def __init__(self, in_channels=2, pretrained=True):
        super().__init__()

        # Load ResNet50 backbone
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)

        # Replace first conv to accept 2-channel input instead of 3
        original_conv = resnet.conv1
        self.encoder_conv1 = nn.Conv2d(
            in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # Initialize new conv from pretrained weights (average RGB channels)
        if pretrained:
            with torch.no_grad():
                w = original_conv.weight
                # Map 3 pretrained channels to 2: blend the third into first two
                self.encoder_conv1.weight[:, 0, :, :] = w[:, 0, :, :] + 0.5 * w[:, 2, :, :]
                self.encoder_conv1.weight[:, 1, :, :] = w[:, 1, :, :] + 0.5 * w[:, 2, :, :]

        # Rest of ResNet encoder (up to layer3 for spatial resolution)
        self.encoder_bn1 = resnet.bn1
        self.encoder_relu = resnet.relu
        self.encoder_maxpool = resnet.maxpool
        self.encoder_layer1 = resnet.layer1
        self.encoder_layer2 = resnet.layer2
        self.encoder_layer3 = resnet.layer3

        self.encoder_pool = nn.AdaptiveAvgPool2d(1)

        # Classification head: takes abs difference of feature vectors
        encoder_dim = 1024  # Output channels of layer3
        self.head = nn.Sequential(
            nn.Linear(encoder_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )

    def encode(self, x):
        """Run encoder on a single patch."""
        x = self.encoder_conv1(x)
        x = self.encoder_bn1(x)
        x = self.encoder_relu(x)
        x = self.encoder_maxpool(x)
        x = self.encoder_layer1(x)
        x = self.encoder_layer2(x)
        x = self.encoder_layer3(x)
        x = self.encoder_pool(x)
        x = x.flatten(1)
        return x

    def forward(self, pre, post):
        """Compute change probability between pre and post patches.

        Parameters
        ----------
        pre : Tensor (B, 2, H, W)
        post : Tensor (B, 2, H, W)

        Returns
        -------
        Tensor (B,) — change probability in [0, 1]
        """
        feat_pre = self.encode(pre)
        feat_post = self.encode(post)

        # Absolute difference captures change magnitude
        diff = torch.abs(feat_pre - feat_post)

        logits = self.head(diff).squeeze(-1)
        return torch.sigmoid(logits)


# Instantiate model and set to evaluation mode
model = SiameseChangeDetector(in_channels=2, pretrained=True).to(device)
model.train(False)  # Switch to inference mode (disables dropout, batchnorm uses running stats)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params:,} parameters")
print(f"Device: {device}")
print("Note: Using ImageNet pretrained ResNet50 backbone (adapted to 2-channel input).")
print("The classification head is randomly initialized — this IS the zero-shot baseline.")

## Load Pre/Post Scenes and Run Inference on Sample Patches

We extract patches at 5 known locations in the Marshall Fire area:
- 2 locations in the burn scar (expected: high change probability)
- 2 locations outside the burn scar (expected: low change probability)
- 1 location at the burn edge (ambiguous)

In [ ]:
# Search for pre and post-fire Sentinel-1 scenes
def search_s1(date_range):
    search = catalog.search(
        collections=["sentinel-1-grd"],
        bbox=AOI_BBOX,
        datetime=date_range,
    )
    items = list(search.items())
    print(f"Found {len(items)} S1 scenes for {date_range}")
    return items


pre_items = search_s1("2021-11-01/2021-12-20")
post_items = search_s1("2022-01-05/2022-01-31")

pre_item = pre_items[0] if pre_items else None
post_item = post_items[0] if post_items else None

if pre_item:
    print(f"\nPre-fire: {pre_item.id} ({pre_item.datetime})")
if post_item:
    print(f"Post-fire: {post_item.id} ({post_item.datetime})")

# Sample locations: (lon, lat, label, description)
SAMPLE_LOCATIONS = [
    (-105.117, 39.952, "burn",    "Louisville — burn scar center"),
    (-105.130, 39.960, "burn",    "Superior — destroyed neighborhood"),
    (-105.145, 39.980, "no_burn", "North — outside fire perimeter"),
    (-105.085, 39.940, "no_burn", "East — unaffected area"),
    (-105.120, 39.965, "edge",    "Burn perimeter edge"),
]

# Extract patches and run inference
results = []

if pre_item and post_item:
    for lon, lat, label, desc in SAMPLE_LOCATIONS:
        print(f"\nExtracting patch at ({lon:.3f}, {lat:.3f}) — {desc}")

        try:
            pre_patch = extract_patch(pre_item, lon, lat, patch_size=64)
            post_patch = extract_patch(post_item, lon, lat, patch_size=64)

            # Normalize to roughly [0, 1] range for the network
            # SAR dB values typically range from -30 to 0
            pre_norm = (pre_patch + 30.0) / 30.0
            post_norm = (post_patch + 30.0) / 30.0

            # Add batch dimension and move to device
            pre_batch = pre_norm.unsqueeze(0).to(device)
            post_batch = post_norm.unsqueeze(0).to(device)

            with torch.no_grad():
                change_prob = model(pre_batch, post_batch).item()

            print(f"  Change probability: {change_prob:.3f}")
            results.append({
                "lon": lon, "lat": lat, "label": label,
                "description": desc, "change_prob": change_prob,
                "pre_patch": pre_patch, "post_patch": post_patch,
            })

        except Exception as e:
            print(f"  Error: {e}")
            results.append({
                "lon": lon, "lat": lat, "label": label,
                "description": desc, "change_prob": None,
                "pre_patch": None, "post_patch": None,
            })

    print("\n=== Summary ===")
    for r in results:
        prob_str = f"{r['change_prob']:.3f}" if r['change_prob'] is not None else "N/A"
        print(f"  {r['label']:8s} | prob={prob_str} | {r['description']}")
else:
    print("Could not find pre/post Sentinel-1 scenes.")

## Visualize Predictions

Top row: VV pre-fire patches. Bottom row: VV post-fire patches.
Each column is one sample location. Border color indicates ground truth label.
Annotated with the predicted change probability score.

In [ ]:
valid_results = [r for r in results if r["pre_patch"] is not None]
n_samples = len(valid_results)

if n_samples > 0:
    fig, axes = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))

    if n_samples == 1:
        axes = axes.reshape(2, 1)

    label_colors = {"burn": "red", "no_burn": "green", "edge": "orange"}

    for i, r in enumerate(valid_results):
        vv_pre = r["pre_patch"][0].numpy()   # VV channel
        vv_post = r["post_patch"][0].numpy()
        border_color = label_colors.get(r["label"], "white")

        # Pre-fire (top row)
        ax = axes[0, i]
        ax.imshow(vv_pre, cmap="gray", vmin=-25, vmax=-5)
        ax.set_title(f"Pre-fire\n{r['label']}", fontsize=10, color=border_color)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)

        # Post-fire (bottom row)
        ax = axes[1, i]
        ax.imshow(vv_post, cmap="gray", vmin=-25, vmax=-5)
        prob = r["change_prob"]
        prob_str = f"P(change)={prob:.3f}" if prob is not None else "N/A"
        ax.set_title(f"Post-fire\n{prob_str}", fontsize=10, color=border_color)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)

        # Location description at bottom
        axes[1, i].set_xlabel(r["description"], fontsize=8, labelpad=10)

    fig.suptitle(
        "Siamese Change Detector — Zero-Shot Predictions\n"
        "(Red border = burn, Green = no burn, Orange = edge)",
        fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

    # Summary statistics
    burn_probs = [r["change_prob"] for r in valid_results
                  if r["label"] == "burn" and r["change_prob"] is not None]
    noburn_probs = [r["change_prob"] for r in valid_results
                    if r["label"] == "no_burn" and r["change_prob"] is not None]

    if burn_probs and noburn_probs:
        print(f"\nBurn locations — mean P(change): {np.mean(burn_probs):.3f}")
        print(f"No-burn locations — mean P(change): {np.mean(noburn_probs):.3f}")
        separation = np.mean(burn_probs) - np.mean(noburn_probs)
        print(f"Separation: {separation:+.3f}")
        if abs(separation) < 0.1:
            print("Very low separation — model needs fine-tuning (expected).")
        elif separation > 0.1:
            print("Some separation even without fine-tuning — ImageNet features capture some change signal.")
        else:
            print("Inverted separation — pretrained features are anti-correlated with damage.")
else:
    print("No valid patches extracted. Check Sentinel-1 data availability.")

## Baseline Performance — Expected ~0.55-0.70 AUC Before Fine-Tuning

The pretrained model uses ImageNet features, which know about visual textures
but nothing about SAR backscatter physics. Expected behavior:

- **Random-ish predictions** — the classification head is randomly initialized,
  so outputs cluster around 0.5
- **Weak separation** — some signal may leak through because destroyed areas
  have different texture statistics, but it is unreliable
- **AUC ~0.55-0.70** — barely above random (0.50) for most locations

After fine-tuning on Marshall Fire training data (notebook 09), we expect:
- AUC > 0.85
- Clear separation between burn and no-burn patches
- Meaningful change probability maps

The gap between this baseline and the fine-tuned model quantifies the value
of domain-specific training data.